# Initial figures

Figures for the meeting. Each figure cell starts with an **INPUTS** block: change the sensors, variable or time window there and rerun that cell only.

- Sensors: Vectors (ADVs) only, from the QC'd files `data/processed/qc/{S}_QC.nc`.
- All times are UTC.
- Figures are saved to `figs/initial/`.

Kernel: **Python (analysiz)** (has granolas).

In [ ]:
# load data:
# 1. buoy data
# 2. tide data from NOAA 1612340 (Honolulu)
# 3. load each sensors .nc file from data/qc/

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import spectrogram, welch
from granolas import sensor_spectra, compute_H0

# ---- paths ----
QC_DIR   = '../data/processed/qc'
META_DIR = '../data/metadata'
FIG_DIR  = '../figs/initial'
os.makedirs(FIG_DIR, exist_ok=True)

# ---- sensors: west -> east, deep -> shallow within a transect ----
# name = V + transect (A-E) + depth (m), e.g. 'VE10' = transect E, 10 m
ALL_SENSORS = ['VA10', 'VB5', 'VC10', 'VC5', 'VD10', 'VD5', 'VE10', 'VE7', 'VE4']

# ---- physics ----
RHO = 1025.0     # seawater density [kg/m^3]
G   = 9.81       # gravity [m/s^2]
FS  = 2.0        # Vector sample rate [Hz]

# 1. buoy: CDIP 238 (Barbers Point). Hs [m], Tp [s], Dp = peak direction waves come FROM [deg true]
buoy = pd.read_csv(f'{META_DIR}/cdip238_waves.csv', index_col='time', parse_dates=True)

# 2. tide: NOAA 1612340 (Honolulu), 6-min water level [m]
tide = pd.read_csv(f'{META_DIR}/noaa_1612340_water_level.csv', index_col=0, parse_dates=True)['Water Level']

# 3. sensors. open_dataset is lazy: nothing big is read until a variable is used (~790 MB per file)
qc = {S: xr.open_dataset(f'{QC_DIR}/{S}_QC.nc') for S in ALL_SENSORS}

for S in ALL_SENSORS:
    t = qc[S].time.values
    print(f'{S:5s} transect {S[1]}  ~{S[2:]:>2s} m   {str(t[0])[:16]} -> {str(t[-1])[:16]}')

## Plot style
- Color = transect: A/B IndianRed, C olive, D cobalt, E near-black.
- Line style = depth: 10 m solid, 7 m dash-dot, 4–5 m dashed.
- Medium-large fonts for slides.

In [ ]:
# claude's fancy plot data, sticking to med-large fonts for readability in a presentation setting:

plt.rcParams.update({
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'axes.labelsize': 14, 'axes.titlesize': 15,
    'xtick.labelsize': 12, 'ytick.labelsize': 12,
    'legend.fontsize': 12, 'legend.frameon': False,
    'lines.linewidth': 1.2,
    'font.family': 'sans-serif',
})

# color by transect (second letter of the sensor name)
T_COLOR = {'A': 'indianred', 'B': 'indianred', 'C': '#708238', 'D': '#0047AB', 'E': '0.15'}

# line style by depth (the number in the sensor name)
DEPTH_LS = {'10': '-', '7': '-.', '5': '--', '4': '--'}

BUOY_COLOR = '0.65'   # light grey for the offshore buoy

# usage:  color=T_COLOR[S[1]], ls=DEPTH_LS[S[2:]]

## Spectra and bulk statistics

For each sensor, over the whole record:

1. **Windows.** The record is cut into `NPERSEG`-sample Hann windows (50 % overlap). Each window is **linearly detrended**, then FFT'd (`scipy.signal.spectrogram`; granolas `sensor_spectra` does the same internally).
2. **Pressure → surface elevation** (granolas `sensor_spectra`, linear theory, bottom-mounted sensor):
$$S_{\eta\eta}(f) = S_{pp}(f)\,\left[\frac{\cosh(kh)}{\rho g}\right]^2, \qquad \omega^2 = gk\tanh(kh)$$
   - The $\cosh(kh)$ factor is capped at 50, and $S_{\eta\eta}=0$ above `F_CUT`.
   - $h$ = mean pressure head + port height above the bed (`zt_z_p_m`). It is smoothed over one window first, because granolas reads $h$ at each window center.
3. **Averaging.** Window spectra are averaged over `AVG` (default 1 h). With `USE_SEG_OK`, windows in hours that failed QC (`seg_ok = False`) are dropped first.
4. **Bulk statistics** (granolas `compute_H0`), over `FMIN`–`FMAX` = 0.04–0.25 Hz:
$$H_s = 4\sqrt{m_0},\quad m_0 = \int_{f_{min}}^{f_{max}} S_{\eta\eta}\,df \qquad\qquad T_p = 1/f_p,\quad f_p = \arg\max_f S_{\eta\eta}$$
5. **Mean wave direction** from the pressure–velocity co-spectra, the same method as `adv_QC` / `adv_QCBulk`. The velocity in phase with pressure points the way the waves travel:
$$\theta_{toward} = \operatorname{atan2}\!\left(\int Co_{pE}\,df,\ \int Co_{pN}\,df\right), \qquad D_m = \theta_{toward} + 180^\circ$$
   - $Co_{pE} = \mathrm{Re}\{P^* U_{east}\}$ and $Co_{pN} = \mathrm{Re}\{P^* V_{north}\}$ are integrated over the same band.
   - $D_m$ is the direction waves come **from** (deg true, nautical), the same convention as the buoy's $D_p$.

Also stored on the same windows (for other spectral figures):
- $S_{pp}$: pressure head, m²/Hz, not depth corrected
- $S_{uu}, S_{vv}, S_{ww}$: shore-normal $u$ (+ onshore), $v$ (alongshore), $w$ (up)

In [ ]:
# take spectra. make sure to detrend. Use scripts from granolas. 
# Have inputs available such as:

# ============ INPUTS ============
SENSORS    = ALL_SENSORS    # e.g. ['VC5'] to test one sensor quickly
NPERSEG    = 1024           # samples per FFT window: 1024 at 2 Hz = 512 s, df = 0.002 Hz
OVERLAP    = 0.5            # window overlap fraction
AVG        = '1h'           # average window spectra over this period before Hs, Tp, Dm
FMIN, FMAX = 0.04, 0.25     # sea-swell band for Hs, Tp, Dm [Hz]
F_CUT      = 0.5            # S_eta set to 0 above this [Hz]
USE_SEG_OK = True           # drop hours that failed QC (seg_ok = False)
# ================================

# compute Hs, Tp, Dm - comment on the equations you use. Use granolas equations. 
# Build an xarray dataset for each sensor's spectra
spec = {}
for S in SENSORS:
    ds  = qc[S]
    df  = ds[['p', 'u', 'v', 'w', 'u_east', 'v_north']].to_dataframe()   # 2 Hz record (~10 M rows)
    z_p = ds.attrs['zt_z_p_m']                                         # pressure port height above bed [m]

    # ---- pressure and depth ----
    df['p_Pa']   = df.p * 1e4                          # dbar -> Pa (what granolas expects)
    df['p_head'] = df.p * 1e4 / (RHO * G)              # dbar -> m of water (hydrostatic head)
    df['h']      = df.p_head + z_p                     # water depth above the bed [m]
    df['h']      = df.h.rolling(NPERSEG, center=True, min_periods=1).mean()   # remove the waves from h

    # ---- surface elevation spectrum S_eta (granolas): Hann window, linear detrend, cosh(kh) transfer ----
    sp = sensor_spectra(df, nperseg=NPERSEG, overlap_frac=OVERLAP,
                        pressure_col='p_Pa', depth_col='h', f_cutoff=F_CUT)

    # ---- other spectra on the exact same windows (same window, overlap and detrend as granolas) ----
    kw = dict(fs=FS, window='hann', nperseg=NPERSEG, noverlap=int(OVERLAP * NPERSEG),
              detrend='linear', scaling='density')
    f, _, Spp = spectrogram(df.p_head.values, mode='psd', **kw)    # pressure head [m^2/Hz]
    f, _, Suu = spectrogram(df.u.values,      mode='psd', **kw)    # cross-shore   [(m/s)^2/Hz]
    f, _, Svv = spectrogram(df.v.values,      mode='psd', **kw)    # alongshore
    f, _, Sww = spectrogram(df.w.values,      mode='psd', **kw)    # vertical

    # complex FFTs -> co-spectra. psd = 2|Z|^2 (one-sided), so co-spectrum = 2 Re(conj(Zp) Zu)
    f, _, Zp = spectrogram(df.p_head.values,  mode='complex', **kw)
    f, _, Ze = spectrogram(df.u_east.values,  mode='complex', **kw)
    f, _, Zn = spectrogram(df.v_north.values, mode='complex', **kw)
    Co_pE = 2 * np.real(np.conj(Zp) * Ze)                          # p - east co-spectrum  [m * m/s / Hz]
    Co_pN = 2 * np.real(np.conj(Zp) * Zn)                          # p - north co-spectrum

    for name, arr in [('Spp', Spp), ('Suu', Suu), ('Svv', Svv), ('Sww', Sww), ('Co_pE', Co_pE), ('Co_pN', Co_pN)]:
        sp[name] = (('frequency', 'time'), arr)
    del df, Zp, Ze, Zn                                              # free memory before the next sensor

    # ---- drop windows in hours that failed QC ----
    if USE_SEG_OK:
        win_hour = pd.DatetimeIndex(sp.time.values).floor('h')                 # clock hour of each window center
        ok = ds.seg_ok.to_series().reindex(win_hour).fillna(False).values      # seg_ok for that hour
        sp = sp.where(xr.DataArray(ok.astype(bool), dims='time'))

    # ---- average window spectra over AVG ----
    sp = sp.resample(time=AVG).mean()

    # ---- Hs, Tp (granolas compute_H0, over FMIN-FMAX) ----
    good = sp.dropna('time', subset=['h_mean'])                    # compute_H0 can't take all-NaN times
    bulk = compute_H0(good, fmin=FMIN, fmax=FMAX)                  # DataFrame: Hs, Tp, H0, h
    bulk = bulk.reindex(sp.time.values)                            # back onto the full time grid
    sp['Hs'] = ('time', bulk.Hs.values)
    sp['Tp'] = ('time', bulk.Tp.values)

    # ---- Dm: direction waves come FROM [deg true] ----
    band   = sp.sel(frequency=slice(FMIN, FMAX))
    toward = np.degrees(np.arctan2(band.Co_pE.sum('frequency'), band.Co_pN.sum('frequency')))
    sp['Dm'] = (toward + 180) % 360
    sp['Dm'] = sp.Dm.where(sp.Hs.notnull())                        # NaN where the hour was dropped

    # ---- labels ----
    sp['Hs'].attrs = {'units': 'm',        'long_name': f'Hs = 4 sqrt(m0), {FMIN}-{FMAX} Hz'}
    sp['Tp'].attrs = {'units': 's',        'long_name': f'peak period, {FMIN}-{FMAX} Hz'}
    sp['Dm'].attrs = {'units': 'deg true', 'long_name': f'mean wave direction, coming FROM, {FMIN}-{FMAX} Hz'}
    sp.attrs.update({'sensor': S, 'transect': S[1], 'depth_m': int(S[2:]),
                     'nperseg': NPERSEG, 'overlap': OVERLAP, 'avg': AVG, 'use_seg_ok': int(USE_SEG_OK)})
    spec[S] = sp

    # check: median Hs here / Hs_SS stored in the QC file (same band, hourly) -> should be ~1
    ratio = (sp.Hs.to_series() / ds.Hs_SS.to_series()).median()
    print(f'{S:5s} {int(sp.Hs.notnull().sum()):5d} x {AVG}   median Hs {float(sp.Hs.median()):.2f} m   '
          f'Tp {float(sp.Tp.median()):.1f} s   Hs/Hs_SS {ratio:.3f}')

# have the option to save it as a netcdf but comment it out for now.
# for S in spec:
#     spec[S].to_netcdf(f'{QC_DIR}/{S}_spec.nc')

## Hs, Tp, Dm: 10 m and 5 m sensors
- The buoy (CDIP 238, offshore of Barbers Point) is in light grey for reference. Its direction is the **peak** direction $D_p$; the ADVs show the band-mean $D_m$.
- **Known head rotations: Dm is suspect after them.** Hs and Tp are not affected.
  - VE4: −28° at 08-25 01:45.
  - VB5: +9° at 08-23 00:25, +71° at 09-06 02:50.
- The data are not corrected here.

In [ ]:
# For both 10m and 5m sensors, make a 3 panel figure with Hs, Tp, and Dm. One figure for 10m, another for 5m. Plot lines for each sensor. For transect E, You may use VE4. Label title clearly

# ============ INPUTS ============
SENSORS_10M = ['VA10', 'VC10', 'VD10', 'VE10']
SENSORS_5M  = ['VB5', 'VC5', 'VD5', 'VE4']
T_START, T_END = '2026-07-19', '2026-09-16'    # e.g. '2026-09-05', '2026-09-12' to zoom on an event
SHOW_BUOY = True
SAVE      = True
# ================================

for group, sensors in [('10 m', SENSORS_10M), ('5 m', SENSORS_5M)]:
    fig, axs = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

    if SHOW_BUOY:
        b = buoy[T_START:T_END]
        axs[0].plot(b.index, b.Hs, color=BUOY_COLOR, lw=1, label='buoy (CDIP 238)')
        axs[1].plot(b.index, b.Tp, '.', ms=3, color=BUOY_COLOR)
        axs[2].plot(b.index, b.Dp, '.', ms=3, color=BUOY_COLOR)

    for S in sensors:
        d = spec[S].sel(time=slice(T_START, T_END))
        color, ls = T_COLOR[S[1]], DEPTH_LS[S[2:]]
        axs[0].plot(d.time, d.Hs, color=color, ls=ls, label=S)
        axs[1].plot(d.time, d.Tp, '.', ms=4, color=color)     # dots: Tp jumps between frequency bins
        axs[2].plot(d.time, d.Dm, '.', ms=4, color=color)     # dots: direction wraps

    axs[0].set_ylabel('Hs (m)')
    axs[1].set_ylabel('Tp (s)')
    axs[2].set_ylabel('Dm (° from, true)')
    axs[0].legend(ncol=5, loc='upper left', markerscale=3)
    axs[0].set_title(f'{group} ADVs: Hs, Tp, Dm ({FMIN}–{FMAX} Hz, {AVG} averages, UTC)', loc='left')

    axs[-1].set_xlim(pd.Timestamp(T_START), pd.Timestamp(T_END))
    axs[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    fig.align_ylabels(axs)
    fig.tight_layout()
    if SAVE:
        fig.savefig(f'{FIG_DIR}/bulk_{group.replace(" ", "")}_{T_START}_{T_END}.png', dpi=200)
    plt.show()

## Transect datasets
One Dataset per transect, with the sensors stacked on a `sensor` dimension. Example: `transects['E'].Seta.sel(sensor='VE10')`.

In [ ]:
# build transect datasets. Each transect is labelled A-E. The sensors title tells you which transect by the second letter in the sensor name. 
# combine spectral datasets from each sensor for each transect

transects = {}
for T in 'ABCDE':
    members = [S for S in spec if S[1] == T]           # sensors on this transect (deep -> shallow)
    if not members:
        continue
    transects[T] = xr.concat([spec[S] for S in members], dim=pd.Index(members, name='sensor'),
                             join='outer', combine_attrs='drop')
    print(T, members)

transects['E']

## Whole-record spectrum, all ADVs
- One spectrum per sensor over the whole record, or over `T_START`–`T_END`.
- It uses Welch (linear detrend, Hann window) instead of one raw FFT. A single FFT of ~10 M points has only 2 degrees of freedom per bin, so it is pure noise. Welch averages the FFTs of `NPERSEG_FULL`-long windows.
  - Bigger `NPERSEG_FULL` gives finer frequency resolution (lower f) but a noisier spectrum.
- Pressure is shown as pressure head (m²/Hz), **not** depth corrected, so the deep sensors drop off faster at high f.

In [ ]:
# spectral figure of the whole time series. Like just fft the whole time series and plot the spectra with all of the ADVs on the same plot. Use loglog axis. Use the same color scheme as the other plots. Make sure to label the axes and add a legend.

# ============ INPUTS ============
SENSORS      = ALL_SENSORS
VARIABLE     = 'p'           # 'p' (pressure head), 'u' (cross-shore), 'v' (alongshore), 'w', 'u_east', 'v_north'
T_START, T_END = None, None  # None = whole record, or e.g. '2026-09-05', '2026-09-12'
NPERSEG_FULL = 2**14         # 16384 samples = 2.3 h windows, df = 1.2e-4 Hz
SHADE_BAND   = (0.04, 0.25)  # light grey band [Hz], or None
SAVE         = True
# ================================

YLABEL = {'p': 'S$_{pp}$ (m$^2$/Hz)',  'u': 'S$_{uu}$ (m$^2$ s$^{-2}$/Hz)', 'v': 'S$_{vv}$ (m$^2$ s$^{-2}$/Hz)',
          'w': 'S$_{ww}$ (m$^2$ s$^{-2}$/Hz)', 'u_east': 'S (east) (m$^2$ s$^{-2}$/Hz)', 'v_north': 'S (north) (m$^2$ s$^{-2}$/Hz)'}

fig, ax = plt.subplots(figsize=(10, 7))
for S in SENSORS:
    x = qc[S][VARIABLE].sel(time=slice(T_START, T_END)).values
    if VARIABLE == 'p':
        x = x * 1e4 / (RHO * G)                       # dbar -> m of water
    f, Sxx = welch(x, fs=FS, window='hann', nperseg=NPERSEG_FULL, detrend='linear')
    ax.loglog(f[1:], Sxx[1:], color=T_COLOR[S[1]], ls=DEPTH_LS[S[2:]], lw=1, label=S)   # f[1:] skips f = 0

if SHADE_BAND:
    ax.axvspan(*SHADE_BAND, color='0.92', zorder=0)
ax.set_xlabel('frequency (Hz)')
ax.set_ylabel(YLABEL[VARIABLE])
window = 'whole record' if T_START is None else f'{T_START} to {T_END}'
ax.set_title(f'{VARIABLE} spectra, {window} (Welch, {NPERSEG_FULL / FS / 3600:.1f} h windows)', loc='left')
ax.legend(ncol=3, loc='lower left')
fig.tight_layout()
if SAVE:
    tag = 'all' if T_START is None else f'{T_START}_{T_END}'
    fig.savefig(f'{FIG_DIR}/spectrum_{VARIABLE}_{tag}.png', dpi=200)
plt.show()